# Task 2: Quantitative Analysis — AAPL

## Objective
Load historical stock price data for Apple (AAPL), compute financial technical indicators,
and visualize the results to understand market behavior.

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')
print('Libraries imported successfully')

## 1. Load and Prepare Data

In [ ]:
import os
STOCK = 'AAPL'
paths = [
    f'../data/raw/{STOCK}.csv',
    rf'c:\KAIM\news-sentiment-analysis\data\raw\{STOCK}.csv',
]
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded: {path}')
        break

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
cols = ['Open', 'High', 'Low', 'Close', 'Volume']
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna()

print(f'Shape: {df.shape}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

## 2. Compute Technical Indicators

In [ ]:
# Moving Averages
df['SMA_20'] = SMAIndicator(df['Close'], window=20).sma_indicator()
df['SMA_50'] = SMAIndicator(df['Close'], window=50).sma_indicator()
df['EMA_20'] = EMAIndicator(df['Close'], window=20).ema_indicator()

# RSI
df['RSI'] = RSIIndicator(df['Close'], window=14).rsi()

# MACD
macd = MACD(df['Close'])
df['MACD'] = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
df['MACD_Hist'] = macd.macd_diff()

# Financial Metrics (PyNance equivalent)
df['Returns'] = df['Close'].pct_change()
df['Cumulative_Return'] = (1 + df['Returns']).cumprod()
df['Volatility_20'] = df['Returns'].rolling(window=20).std() * np.sqrt(252)

annualized_vol = df['Returns'].std() * np.sqrt(252)
print(f'Annualized Volatility: {annualized_vol:.4f}')
print(f'Total Cumulative Return: {df["Cumulative_Return"].iloc[-1]:.2f}x')
print(df[['Close','SMA_20','SMA_50','EMA_20','RSI','MACD']].tail(5).round(2))

## 3. Visualization — Price & Moving Averages

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Close'], label='Close Price', linewidth=1, color='black')
plt.plot(df['Date'], df['SMA_20'], label='SMA 20', linestyle='--', color='blue')
plt.plot(df['Date'], df['SMA_50'], label='SMA 50', linestyle='--', color='orange')
plt.plot(df['Date'], df['EMA_20'], label='EMA 20', linestyle=':', color='green')
plt.title(f'{STOCK} — Price with Moving Averages', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig(f'{STOCK}_price_ma.png', dpi=150, bbox_inches='tight')
plt.show()
print('Price & MA plot saved!')

## 4. RSI — Overbought & Oversold Conditions

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df['Date'], df['RSI'], color='purple', linewidth=1)
plt.axhline(70, color='red', linestyle='--', label='Overbought (70)')
plt.axhline(30, color='green', linestyle='--', label='Oversold (30)')
plt.fill_between(df['Date'], df['RSI'], 70, where=(df['RSI'] >= 70), alpha=0.3, color='red')
plt.fill_between(df['Date'], df['RSI'], 30, where=(df['RSI'] <= 30), alpha=0.3, color='green')
plt.title(f'{STOCK} — RSI (14)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('RSI')
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.savefig(f'{STOCK}_rsi.png', dpi=150, bbox_inches='tight')
plt.show()
print('RSI plot saved!')

## 5. MACD — Momentum & Trend Reversals

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df['Date'], df['MACD'], label='MACD', color='blue', linewidth=1)
plt.plot(df['Date'], df['MACD_Signal'], label='Signal Line', color='orange', linewidth=1)
plt.bar(df['Date'], df['MACD_Hist'],
        color=['green' if v >= 0 else 'red' for v in df['MACD_Hist']], alpha=0.4, width=1)
plt.axhline(0, color='black', linewidth=0.5)
plt.title(f'{STOCK} — MACD', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('MACD')
plt.legend()
plt.tight_layout()
plt.savefig(f'{STOCK}_macd.png', dpi=150, bbox_inches='tight')
plt.show()
print('MACD plot saved!')

## 6. Summary Statistics & Observations

### Data Preparation Steps:
- Loaded historical AAPL stock price data from CSV
- Converted Date column to datetime format
- Ensured correct numeric types for OHLCV columns
- Handled missing values using dropna()

### Indicators Computed:
- **SMA 20 & 50:** Simple Moving Averages for short and medium-term trends
- **EMA 20:** Exponential Moving Average — reacts faster to price changes
- **RSI 14:** Momentum oscillator — identifies overbought (>70) and oversold (<30) conditions
- **MACD (12,26,9):** Detects momentum shifts and potential trend reversals
- **Daily Returns & Volatility:** Financial metrics for risk assessment

### Observations:
- RSI spikes above 70 were associated with short-term price reversals
- MACD crossovers aligned with key trend changes and momentum shifts
- SMA 50 frequently acted as support/resistance during trending market phases

In [ ]:
print(f'=== {STOCK} Summary Statistics ===')
print(df[['Close', 'SMA_20', 'SMA_50', 'RSI', 'MACD']].describe().round(2))